# Phase 4 — Agent A4 : Discourse Analyser (PRD v3.0 — CoT + ToT conditionnel + Tools)
**YouTube Quality Analyzer** | Phase 4 — Implémentation

Ce notebook démontre l'agent A4 (`agents/a4_discourse.py`) en version **PRD v3.0** :
- **Tools pré-LLM** : `compute_text_stats` + `detect_argumentative_markers`
- **CoT standard** si score hors zone d'ambiguïté [0.35–0.65]
- **ToT déclenché** si score CoT ∈ [0.35–0.65] (3 branches : Content Depth / Discussion Quality / Epistemic Value)
- 3 dimensions : Informativité, Argumentation, Constructivité → `Score_Discours` [0-100]
- Identification des `high_quality_indices` pour A7
- Pipeline anti-hallucination : `safe_llm_call` → `DiscourseValidator` → retry ×3

> **Double mock requis** : `safe_llm_call` importe `get_llm` localement depuis `models.llm_loader`.  
> Patcher **à la fois** `agents.a4_discourse.get_llm` (garde) ET `models.llm_loader.get_llm` (safe_llm_call).

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath(".."))

from unittest.mock import MagicMock, patch
from agents.a4_discourse import a4_discourse

## 1. Corpus de test

In [ ]:
CLEANED_COMMENTS = [
    {"text": "The approach mirrors what Goodfellow describes in Deep Learning ch.9 — batch norm absent here explains training instability at 12:34.", "cleaned_text": "the approach mirrors what goodfellow describes in deep learning ch.9 — batch norm absent here explains training instability at 12:34."},
    {"text": "Great video!", "cleaned_text": "great video!"},
    {"text": "lol first", "cleaned_text": "lol first"},
    {"text": "If X implies Y then Z cannot hold simultaneously — your proof at 18:45 contradicts 5:20.", "cleaned_text": "if x implies y then z cannot hold simultaneously — your proof at 18:45 contradicts 5:20."},
    {"text": "Check my channel for more content!", "cleaned_text": "check my channel for more content!"},
]

STATE = {"cleaned_comments": CLEANED_COMMENTS}
print(f"{len(CLEANED_COMMENTS)} commentaires (indices 0–{len(CLEANED_COMMENTS)-1})")

## 2. A4 avec LLM mocké — discours de qualité

In [ ]:
# v3.0 — schéma JSON CoT (reasoning + discourse_score + tot_used)
mock_response = MagicMock()
mock_response.content = json.dumps({
    "reasoning": (
        "Thought 1: Informativeness — comments reference Goodfellow ch.9 and batch norm → high. "
        "Thought 2: Argumentation — logical proof contradiction cited → solid. "
        "Thought 3: Constructiveness — substantive feedback > superficial. "
        "Thought 4: HQ indices — [0, 3] exceed 0.7 threshold."
    ),
    "informativeness":       0.78,
    "argumentation":         0.65,
    "constructiveness":      0.80,
    "discourse_score":       0.743,   # moyenne des 3 dimensions
    "high_quality_indices":  [0, 3],
    "rationale": "Two comments add substantive evidence; others are superficial.",
    "tot_used":  False,
})

mock_llm = MagicMock()
mock_llm.invoke.return_value = mock_response

# Double mock : garde du nœud + safe_llm_call (import local)
with patch("agents.a4_discourse.get_llm", return_value=mock_llm), \
     patch("models.llm_loader.get_llm",   return_value=mock_llm):
    result = a4_discourse(STATE)

d = result["discourse"]
print(f"Informativité    : {d['informativeness']:.1f} / 100")
print(f"Argumentation    : {d['argumentation']:.1f} / 100")
print(f"Constructivité   : {d['constructiveness']:.1f} / 100")
print(f"Score_Discours   : {d['discourse_score']:.2f} / 100")
print(f"ToT utilisé      : {d.get('tot_used')}")
print(f"High-quality idx : {d['high_quality_indices']}")
print(f"hallucination_flags : {result.get('hallucination_flags', [])}")
print()
print("Commentaires haute qualité sélectionnés pour A7 :")
for i in d["high_quality_indices"]:
    if i < len(CLEANED_COMMENTS):
        print(f"  [{i}] {CLEANED_COMMENTS[i]['text']!r}")

## 3. Fallback — LLM indisponible

In [ ]:
# Fallback — LLM indisponible (patch garde seul suffit pour bloquer le nœud)
with patch("agents.a4_discourse.get_llm", return_value=None):
    fallback = a4_discourse(STATE)

d = fallback["discourse"]
print(f"Fallback discourse_score     : {d['discourse_score']}")
print(f"Fallback informativeness     : {d['informativeness']}")
print(f"Fallback argumentation       : {d['argumentation']}")
print(f"Fallback constructiveness    : {d['constructiveness']}")
print(f"Fallback high_quality_indices: {d['high_quality_indices']}")
print(f"Fallback tot_used            : {d.get('tot_used')}")

# v3.0 : fallback heuristique basé sur compute_text_stats + detect_argumentative_markers
# → score calculé dynamiquement, high_quality_indices toujours vide
assert d["high_quality_indices"] == [], f"high_quality_indices doit être vide en fallback : {d['high_quality_indices']}"
assert 0.0 <= d["discourse_score"] <= 100.0, f"Score hors [0-100] : {d['discourse_score']}"
print("\nFallback v3.0 conforme (heuristique tools-based).")

## Résumé A4 — PRD v3.0

| Comportement | Résultat |
|---|---|
| **Tool 1** : `compute_text_stats` | word_count, unique_word_ratio, question_count… injectés |
| **Tool 2** : `detect_argumentative_markers` | argumentation_score [0-1] injecté |
| **CoT standard** | Thought 1→2→3→4 → Result |
| **ToT conditionnel** | Déclenché si CoT score ∈ [0.35–0.65] (zone ambiguë) |
| 3 dimensions [0,1] → [0,100] | Normalisation confirmée |
| `discourse_score` fourni par le LLM | Validé par `DiscourseValidator` |
| `tot_used` | `True` si ToT déclenché, sinon `False` |
| `hallucination_flags` | `"tot_triggered"` ajouté si ToT activé |
| `high_quality_indices` | Passés à A7 pour le topic matching |
| LLM indisponible | Fallback **heuristique tools-based** — score dynamique, indices vides |
| Double mock requis | `agents.a4_discourse.get_llm` + `models.llm_loader.get_llm` |

> **Poids w2 = 0.40** — dimension la plus influente dans Score_Global (H3)